# 08 · Recursion with matrices and vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb)

*Part IV · demo · 10 min*

> 🇪🇸 **Recursión con matrices y vectores** — Entender una recurrencia como una actualización de estado repetida, observar cuándo domina una dirección propia y comprobar qué ocurre cuando un pronóstico reutiliza sus propias predicciones.

Treat recursion as repeated state updates, connect repeated multiplication with dominant eigen-directions, and test recursive forecasting on real airline-passenger data.

## What you will be able to do

- Write a recurrence as a repeated state update `x[t+1] = A @ x[t]`.
- Explain why repeated multiplication can align a state with a dominant eigenvector.
- Use a controlled synthetic matrix to see how the eigenvalue ratio controls convergence speed.
- Fit a real autoregressive model with the pseudoinverse and feed its own predictions back in.
- Diagnose why recursive forecast error can compound with horizon.

> 🇪🇸 **Lo que podrás hacer:**

> - Escribir una recurrencia como una actualización repetida del estado `x[t+1] = A @ x[t]`.
> - Explicar por qué las multiplicaciones repetidas pueden alinear un estado con un autovector dominante.
> - Usar una matriz sintética controlada para observar cómo la razón entre autovalores controla la velocidad de convergencia.
> - Ajustar un modelo autorregresivo real con la pseudoinversa y reutilizar sus propias predicciones como entradas.
> - Diagnosticar por qué el error de un pronóstico recursivo puede acumularse con el horizonte.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Enable ipywidgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

FLIGHTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"
flights = pd.read_csv(FLIGHTS)

y = flights["passengers"].to_numpy(float)
labels = (
    flights["year"].astype(str)
    + "-"
    + flights["month"].astype(str).str[:3]
).to_numpy()

rng = np.random.default_rng(0)

print("real months / meses reales:", len(y))
print("range / periodo:", labels[0], "→", labels[-1])
print("passengers min/max:", int(y.min()), int(y.max()))
print("interactive charts: Plotly + ipywidgets enabled")

## Why this matters

A recurrence does not need a mysterious new operation. It can be as simple as reusing the **same update rule** over and over:

`x[t+1] = A x[t]`

The important idea is that the output at one step becomes the input to the next. That creates **state**.

This same pattern appears in several places:

- a second-order recurrence such as Fibonacci can be rewritten as a matrix state update;
- power iteration repeatedly applies a matrix until one direction dominates;
- recursive forecasting predicts the next value, then feeds that prediction back as if it were observed.

The mathematics is similar, but the consequences differ: repetition can reveal structure, or it can amplify error.

### How to use the folded solutions / Cómo usar las soluciones plegadas

Each exercise has a **Solution / Solución** cell that is intentionally closed. First try the `TODO`; then open the solution to compare your reasoning with a reference implementation.

> 🇪🇸 Una recurrencia reutiliza la misma regla de actualización y convierte la salida de un paso en la entrada del siguiente. Ese patrón aparece en Fibonacci, en la iteración de potencias y en el pronóstico recursivo. La repetición puede revelar una dirección dominante, pero también puede propagar errores.
>
> Cada ejercicio tiene una celda **Solution / Solución** cerrada a propósito. Primero intenta resolver el `TODO`; después abre la solución para comparar tu razonamiento con una implementación de referencia.

### Learning cycle: Predict → Run → Explain

Before every exercise, predict what repeated application should do, run the update, and then explain what changed and why.

> 🇪🇸 **Predice → Ejecuta → Explica:** antes de cada ejercicio anticipa qué debería hacer la aplicación repetida, ejecuta la actualización y explica qué cambió y por qué.

## Exercise 1 — turn a recurrence into a state update

Fibonacci looks scalar:

`f[n+1] = f[n] + f[n-1]`

but it becomes a two-dimensional state:

`[f[n+1], f[n]]ᵀ = F [f[n], f[n-1]]ᵀ`

with

`F = [[1, 1], [1, 0]]`.

This is our cleanest example of **recursion as repeated matrix multiplication**.

### What should you try?

1. Start from `[1, 0]`.
2. Apply the same matrix 10 times.
3. Compare the loop with `np.linalg.matrix_power`.
4. Move the **Steps / Pasos** slider and watch the state grow.

> 🇪🇸 Fibonacci parece una recurrencia escalar, pero puede escribirse como un estado bidimensional actualizado siempre por la misma matriz. Prueba el `TODO`, compara el ciclo con `matrix_power` y usa el slider para observar cómo evoluciona el estado.

In [ ]:
# TODO
# 1. Define F = [[1, 1], [1, 0]] and v0 = [1, 0].
# 2. Apply F repeatedly for 10 steps.
# 3. Compare the loop result with np.linalg.matrix_power(F, 10) @ v0.
# 4. Predict which entry contains Fibonacci(10).

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
F = np.array([[1, 1], [1, 0]], dtype=object)
v0 = np.array([1, 0], dtype=object)

v = v0.copy()
for _ in range(10):
    v = F @ v

via_power = np.linalg.matrix_power(F, 10) @ v0

print("loop / ciclo:", v)
print("matrix_power:", via_power)
print("Fibonacci(10):", int(v[1]))
print("same result / mismo resultado:", np.array_equal(v, via_power))

steps_slider = widgets.IntSlider(
    value=10, min=1, max=25, step=1,
    description="Steps / Pasos:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def explore_fibonacci(steps):
    state = np.linalg.matrix_power(F, steps) @ v0
    seq = []
    s = v0.copy()
    for k in range(steps + 1):
        seq.append((k, int(s[0]), int(s[1])))
        s = F @ s

    df = pd.DataFrame(seq, columns=["step", "f_next", "f_current"])
    print(
        f"step/paso={steps} | state/estado={state.tolist()} | "
        f"Fibonacci({steps})={int(state[1])}"
    )

    fig = px.line(
        df,
        x="step",
        y=["f_next", "f_current"],
        markers=True,
        title="Repeated state update / Actualización repetida del estado",
        labels={"value": "state value / valor", "step": "step / paso"},
    )
    fig.update_layout(
        height=290,
        width=620,
        margin=dict(l=55, r=20, t=55, b=50),
        legend_title_text="state / estado",
    )
    fig.show()

fib_output = widgets.interactive_output(
    explore_fibonacci,
    {"steps": steps_slider},
)

display(widgets.VBox([steps_slider, fib_output]))

## Exercise 2 — when repeated multiplication chooses a direction

Power iteration uses the same recursive skeleton:

`x[t+1] = A x[t]`, followed by normalization.

For a suitable matrix, repeated multiplication tends to align the vector with the eigenvector associated with the largest-magnitude eigenvalue.

Here we intentionally use a **small synthetic 2×2 matrix**. The goal is not to pretend it is real data; the controlled matrix lets us change the eigenvalue gap and isolate exactly what controls convergence speed.

### What should you try?

1. Run power iteration for 1, 2, 5, 10 and 50 steps.
2. Compare the estimated direction with `np.linalg.eig`.
3. Use the **λ₂ / λ₁** slider to make the two eigenvalues closer.
4. Predict what happens as the ratio approaches 1.

> 🇪🇸 Aquí usamos deliberadamente una matriz sintética `2×2` porque queremos aislar un mecanismo matemático: la velocidad de convergencia depende de qué tan dominante sea el autovalor principal. Acerca `λ₂/λ₁` a 1 y observa cómo la recursión tarda más en alinearse con la dirección dominante.

In [ ]:
# TODO
# 1. Implement power iteration with normalization after every multiplication.
# 2. Track the angle between the current vector and the dominant eigenvector.
# 3. Compare a matrix with a clear spectral gap against one whose eigenvalues
#    are almost equal.

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
def power_trace(M, steps=40, seed=0):
    eigvals, eigvecs = np.linalg.eig(M)
    idx = int(np.argmax(np.abs(eigvals)))
    dominant = eigvecs[:, idx].real
    dominant /= np.linalg.norm(dominant)

    v = np.random.default_rng(seed).standard_normal(M.shape[0])
    v /= np.linalg.norm(v)

    rows = []
    for step in range(1, steps + 1):
        v = M @ v
        v /= np.linalg.norm(v)

        alignment = abs(float(np.dot(v, dominant)))
        alignment = min(1.0, max(0.0, alignment))
        angle = np.degrees(np.arccos(alignment))
        rq = float(v @ M @ v)

        rows.append({
            "step": step,
            "angle_deg": angle,
            "rayleigh": rq,
        })

    return pd.DataFrame(rows), eigvals, dominant

ratio_slider = widgets.FloatSlider(
    value=0.40,
    min=0.10,
    max=0.99,
    step=0.01,
    description="λ₂ / λ₁:",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "80px"},
)

def explore_power_iteration(ratio):
    # Controlled symmetric matrix with eigenvalues 5 and 5*ratio.
    Q = np.array([
        [np.cos(np.pi / 6), -np.sin(np.pi / 6)],
        [np.sin(np.pi / 6),  np.cos(np.pi / 6)],
    ])
    D = np.diag([5.0, 5.0 * ratio])
    M = Q @ D @ Q.T

    trace, eigvals, dominant = power_trace(M, steps=40, seed=0)

    print("eigenvalues / autovalores:", np.round(np.sort(eigvals)[::-1], 3))
    print("ratio λ₂/λ₁:", f"{ratio:.2f}")
    print("final angle / ángulo final:", f"{trace['angle_deg'].iloc[-1]:.4f}°")

    fig = px.line(
        trace,
        x="step",
        y="angle_deg",
        markers=True,
        title="Angle to dominant eigenvector / Ángulo al autovector dominante",
        labels={
            "step": "iteration / iteración",
            "angle_deg": "angle (degrees) / ángulo (grados)",
        },
    )
    fig.update_layout(
        height=300,
        width=620,
        margin=dict(l=55, r=20, t=55, b=50),
    )
    fig.show()

power_output = widgets.interactive_output(
    explore_power_iteration,
    {"ratio": ratio_slider},
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Spectral-gap explorer / Explorador de brecha espectral:</b> "
            "move λ₂/λ₁ toward 1 and watch convergence slow down. / "
            "acerca λ₂/λ₁ a 1 y observa cómo se ralentiza la convergencia."
        ),
        ratio_slider,
        power_output,
    ])
)

## Exercise 3 — recursive forecasting on real airline traffic

Now recursion becomes operational.

The dataset contains **144 real monthly passenger counts from 1949 to 1960**. We hold out the final 12 months, fit an autoregressive model to the earlier observations, and then forecast recursively.

For a window of length `p`, the model learns:

`next month = bias + w₁·previous value + ... + wₚ·value p months ago`

The weights come from the pseudoinverse, linking this notebook directly to Section 07.

The crucial recursive step is this: after predicting one month, that prediction is appended to history and becomes an input for the next prediction.

### What should you try?

1. Fit with `p = 12` and forecast the held-out 12 months.
2. Compare predictions with the real values the model never saw during fitting.
3. Move **Window / Ventana** from 3 to 24.
4. Move **Horizon / Horizonte** farther into the future and watch recursive uncertainty grow.
5. Explain why `p = 12` is meaningful for monthly seasonal data.

> 🇪🇸 Ahora usamos 144 observaciones mensuales reales. Reservamos los últimos 12 meses, ajustamos un modelo autorregresivo con la pseudoinversa y pronosticamos de manera recursiva. Cada predicción pasa a formar parte de la historia usada para producir la siguiente. Cambia la ventana y el horizonte para observar cómo la estructura estacional y los errores se propagan.

In [ ]:
# TODO
# 1. Hold out the final 12 real months.
# 2. Fit an autoregressive model with p=12 using np.linalg.pinv.
# 3. Forecast the 12 held-out months recursively.
# 4. Compute MAPE.
# 5. Try p=3 and explain what seasonal information is lost.

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
def fit_ar(series, p):
    rows = np.array(
        [series[i:i+p] for i in range(len(series) - p)],
        dtype=float,
    )
    X_ar = np.column_stack([np.ones(len(rows)), rows])
    target = series[p:]
    w_ar = np.linalg.pinv(X_ar) @ target
    return w_ar

def recursive_forecast(history, w_ar, p, steps):
    hist = list(np.asarray(history, dtype=float))
    out = []

    for _ in range(steps):
        nxt = float(w_ar[0] + np.dot(w_ar[1:], hist[-p:]))
        hist.append(nxt)
        out.append(nxt)

    return np.asarray(out)

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return np.mean(np.abs(predicted - actual) / actual)

holdout = 12
y_train = y[:-holdout]
y_test = y[-holdout:]

w12 = fit_ar(y_train, 12)
pred12 = recursive_forecast(y_train, w12, 12, holdout)

print("train months / meses entrenamiento:", len(y_train))
print("held-out months / meses reservados:", len(y_test))
print("p=12 holdout MAPE:", f"{mape(y_test, pred12):.1%}")

window_slider = widgets.IntSlider(
    value=12, min=3, max=24, step=1,
    description="Window / Ventana:",
    continuous_update=False,
    style={"description_width": "115px"},
)

horizon_slider = widgets.IntSlider(
    value=12, min=6, max=36, step=6,
    description="Horizon / Horizonte:",
    continuous_update=False,
    style={"description_width": "125px"},
)

def explore_recursive_forecast(window, horizon):
    # Fit only on data before the final 12 real months.
    train = y[:-12]
    w_ar = fit_ar(train, window)

    # Start recursive forecast at the end of training.
    pred = recursive_forecast(train, w_ar, window, horizon)

    start = len(train)
    future_idx = np.arange(start, start + horizon)

    available_actual = y[start:min(start + horizon, len(y))]
    actual_idx = np.arange(start, start + len(available_actual))

    fig = go.Figure()

    context_start = max(0, start - 36)
    fig.add_trace(
        go.Scatter(
            x=np.arange(context_start, start),
            y=y[context_start:start],
            mode="lines",
            name="training context / contexto",
        )
    )

    if len(available_actual):
        fig.add_trace(
            go.Scatter(
                x=actual_idx,
                y=available_actual,
                mode="lines+markers",
                name="held-out actual / real reservado",
            )
        )

    fig.add_trace(
        go.Scatter(
            x=future_idx,
            y=pred,
            mode="lines+markers",
            name="recursive forecast / pronóstico recursivo",
        )
    )

    fig.add_vline(x=start - 0.5, line_dash="dash")

    fig.update_layout(
        title=(
            f"Recursive forecast — window={window}, horizon={horizon} / "
            f"ventana={window}, horizonte={horizon}"
        ),
        xaxis_title="month index / índice mensual",
        yaxis_title="passengers / pasajeros",
        height=330,
        width=680,
        margin=dict(l=55, r=20, t=60, b=50),
        legend=dict(orientation="h", y=-0.23),
    )
    fig.show()

    comparable = min(len(available_actual), len(pred))
    if comparable:
        err = mape(available_actual[:comparable], pred[:comparable])
        print(
            f"MAPE on {comparable} real held-out months / "
            f"MAPE en {comparable} meses reales reservados: {err:.1%}"
        )

    if horizon > 12:
        print(
            "EN: after month 12, the chart is beyond the dataset; "
            "there is no real target here to validate against."
        )
        print(
            "ES: después del mes 12, el pronóstico queda fuera del conjunto; "
            "no existe un valor real aquí para validarlo."
        )

    print(
        "EN: every predicted month becomes an input to the next prediction."
    )
    print(
        "ES: cada mes predicho se convierte en entrada de la siguiente predicción."
    )

forecast_output = widgets.interactive_output(
    explore_recursive_forecast,
    {
        "window": window_slider,
        "horizon": horizon_slider,
    },
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Recursive forecast explorer / Explorador de pronóstico recursivo:</b> "
            "change memory length and forecast horizon. / "
            "cambia la longitud de memoria y el horizonte."
        ),
        window_slider,
        horizon_slider,
        forecast_output,
    ])
)

## What just happened

You used the same idea — **reuse the current state to create the next state** — in three settings.

1. **Fibonacci:** the state `[f[n], f[n-1]]` was updated by the same matrix at every step. `matrix_power` compressed many recursive updates into one matrix power.
2. **Power iteration:** repeated multiplication amplified the dominant eigendirection. When `λ₂/λ₁` moved closer to 1, convergence slowed because the dominant direction was less dominant.
3. **Real airline forecasting:** the pseudoinverse fitted one-step linear dynamics from real observations, then recursion fed predictions back into the model. That made long-horizon errors capable of compounding.

### The sentence to remember

> **Recursion is repeated state update: the next input contains the previous output.**

That is also the skeleton behind recurrent neural networks: the same parameters are reused across sequence steps, while a hidden state carries information forward. Modern sequence models often use more elaborate mechanisms, but this state-update view is the essential bridge.

> 🇪🇸 Usaste la misma idea — **reutilizar el estado actual para construir el siguiente** — en tres contextos. Fibonacci mostró la actualización matricial repetida; la iteración de potencias mostró cómo una dirección propia puede dominar; y el pronóstico real mostró cómo una predicción puede convertirse en entrada y propagar error.
>
> **Frase para recordar:** la recursión es una actualización repetida del estado: la siguiente entrada contiene la salida anterior.
>
> Esta es también la estructura básica de una red neuronal recurrente: los mismos parámetros se reutilizan a lo largo de la secuencia mientras un estado oculto transporta información hacia adelante.

---

## Done with this section

> 🇪🇸 **Fin de esta sección.**

Next up: **09 · Convolution and deconvolution** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)